# 02 — Correlation Analysis

Examines the correlation structure of candidate predictors within each analysis arm,
separately, before hypothesis testing and regression:

- **Environmental arm:** the 9 ward colonization-pressure variables (`{pathogen}_cp`) —
  both their relationship with MRSA acquisition and with each other (multicollinearity).
- **Patient arm:** the antibiotic-class course-count variables (`{abx_class}_0_60`) —
  same two questions.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import pointbiserialr

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

env = load_processed("environmental_mrsa.csv")
pat = load_processed("patient_mrsa.csv")

## Environmental: colonization pressure correlations

In [ ]:
cp_cols = [c for c in env.columns if c.endswith("_cp")]

corr = env[cp_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Spearman correlation among ward colonization-pressure variables")
plt.tight_layout()

In [ ]:
# Point-biserial correlation of each CP variable with case/control status
results = []
for c in cp_cols:
    r, p = pointbiserialr(env["group_binary"], env[c])
    results.append({"variable": c, "r": r, "p": p})
pd.DataFrame(results).sort_values("r", ascending=False)

## Patient: antibiotic exposure correlations

In [ ]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]

corr = pat[abx_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
ax.set_title("Spearman correlation among antibiotic class course counts")
plt.tight_layout()

In [ ]:
results = []
for c in abx_cols:
    r, p = pointbiserialr(pat["group_binary"], pat[c])
    results.append({"variable": c, "r": r, "p": p})
pd.DataFrame(results).sort_values("r", ascending=False)

## Notes for the regression stage

Flag any predictor pairs with |Spearman r| > ~0.7 here — they're multicollinearity
candidates worth checking with VIF in `04a`/`04b` rather than dropping outright.